In [2]:
import pandas as pd
import numpy as np
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings
from datasets import Dataset
from transformers import AutoTokenizer, AutoModel
from transformers import LlamaTokenizer, LlamaModel
from sklearn.preprocessing import normalize
import torch

### Step1: Model Choice

In [3]:
# Model selection
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
model = AutoModel.from_pretrained("google-bert/bert-base-uncased")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move the model to the device (GPU if available, otherwise CPU)
model = model.to(device)

### Step2: Load any relevant reference for RAG

In [4]:
# Load the ICD11 guildlines
path_guideline = "/teamspace/studios/this_studio/ICD11-Psych/Team_5/code/data/Mood_Anxiety_Stress.pdf"
loader = PyPDFLoader(path_guideline)
pages = loader.load()

# Step 2: Manually assign page numbers to the metadata
for i, page in enumerate(pages):
    page.metadata['page_number'] = i + 1

In [5]:
print(f"Number of pages: {len(pages)}")
print(f"Length of example page: {len(pages[1].page_content)}")
print("Content of example page:", pages[1].page_content)

Number of pages: 114
Length of example page: 2294
Content of example page: Clinical Descriptions and Diagnostic Requirements for ICD-11 Mental, Behavioural or Neurodevelopmental Disorders212
CDDR are also provided for GA34.41 Premenstrual dysphoric disorder in the section on 
depressive disorders. Premenstrual dysphoric disorder is classified in the grouping of premenstrual 
disturbances in Chapter 16 on diseases of the genitourinary system, but is cross-listed here for 
reference.
A category for mood disorders that do not fit the descriptions for any of the above categories is 
also provided (other specified), as is a category for use when it is not possible to make a more 
definitive diagnosis (unspecified):
6A8Y Other specified mood disorder
6A8Z Mood disorder, unspecified
The sections that follow describe the characteristics of mood episodes. After that, the CDDR are 
provided for the diagnostic categories of mood disorders.
Mood episode descriptions
Depressive episode
Essential (r

### Step 3: Chunk the reference

In [6]:
# Split the pages in chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=100)

chunks = splitter.split_documents(pages)
print(f"Number of chunks: {len(chunks)}")
print(f"Length of a chunk: {len(chunks[1].page_content)}")
print("Content of a chunk:", chunks[1].page_content)

Number of chunks: 269
Length of a chunk: 1434
Content of a chunk: Clinical Descriptions and Diagnostic Requirements for ICD-11 Mental, Behavioural or Neurodevelopmental Disorders212
CDDR are also provided for GA34.41 Premenstrual dysphoric disorder in the section on 
depressive disorders. Premenstrual dysphoric disorder is classified in the grouping of premenstrual 
disturbances in Chapter 16 on diseases of the genitourinary system, but is cross-listed here for 
reference.
A category for mood disorders that do not fit the descriptions for any of the above categories is 
also provided (other specified), as is a category for use when it is not possible to make a more 
definitive diagnosis (unspecified):
6A8Y Other specified mood disorder
6A8Z Mood disorder, unspecified
The sections that follow describe the characteristics of mood episodes. After that, the CDDR are 
provided for the diagnostic categories of mood disorders.
Mood episode descriptions
Depressive episode
Essential (required) 

### Step 4 Get the representation for each of the chunk

In [7]:
# Storing the chunks in a vector store
# pip install -U :class:`~langchain-ollama
# from :class:`~langchain_ollama import OllamaEmbeddings`
# embeddings = OllamaEmbeddings(model=model)
titles = [doc.metadata['source'] for doc in chunks]
texts = [doc.page_content for doc in chunks]
pages = [doc.metadata['page_number'] for doc in chunks]

# Create a Dataset object
data = {'title': titles, 'text': texts, 'page_number': pages}
dataset = Dataset.from_dict(data)

In [8]:
# Get the embeddings
def embed_text(batch):
    inputs = tokenizer(batch['text'], return_tensors='pt', padding=True, truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
    return {'embeddings': [emb for emb in embeddings]}

# Map the function over the dataset to add the embeddings
dataset = dataset.map(embed_text, batched=True, batch_size=8)

Map:   0%|          | 0/269 [00:00<?, ? examples/s]

### Step 5 Calculate the simlarity between the prompt and each of the chunk, extract the topk relevant chunks. Then concatenate the query and the retrieved texts

In [9]:
# Extract embeddings and titles/texts from the dataset
titles = dataset['title']
texts = dataset['text']
pages = dataset['page_number']
embeddings = np.stack(dataset['embeddings'])

# Normalize embeddings for cosine similarity
normalized_embeddings = normalize(embeddings, axis=1)

# Convert embeddings to PyTorch tensor and move to GPU
doc_embeddings = torch.tensor(normalized_embeddings, dtype=torch.float).to(device)

# Define the retrieval function
def retrieve_top_k(query, k=5):
    # Tokenize and compute query embedding
    inputs = tokenizer(query, return_tensors='pt', padding=True, truncation=True, max_length=512).to(device)
    with torch.no_grad():
        query_embedding = model(**inputs).last_hidden_state.mean(dim=1)  # Shape: (1, hidden_size)

    # Normalize the query embedding
    query_embedding = normalize(query_embedding.cpu().numpy(), axis=1)
    query_embedding = torch.tensor(query_embedding, dtype=torch.float).to(device)

    # Compute cosine similarity between the query embedding and document embeddings
    similarities = torch.nn.functional.cosine_similarity(query_embedding, doc_embeddings)  #

    # Get the top K most similar document indices
    top_k_indices = torch.topk(similarities, k=k).indices.cpu().numpy()

    # Return the top K titles and texts
    top_k_titles = [titles[i] for i in top_k_indices]
    top_k_texts = [texts[i] for i in top_k_indices]
    top_k_pages = [pages[i] for i in top_k_indices]
    return top_k_titles, top_k_texts, top_k_pages
    
# Concatenate the query and the retrieved texts
def concatenate_texts(instr, query, texts):
    instruction = instr + query + "\n\nTry to do the diagnosis based on the following reference(s):\n"
    # Concatenate the instruction and retrieved texts
    return instruction + " ".join(texts)

In [10]:
# Example query
query = "I am sad, I am unhappy."
top_k_titles, top_k_texts, top_k_pages = retrieve_top_k(query, k=5)

print("Top 5 Retrieved Documents:")
for i, (title, text, page) in enumerate(zip(top_k_titles, top_k_texts, top_k_pages)):
    print(f"{i+1}. Title: {title}\n   Page: {page}\n   Content: {text[:300]}...\n")

Top 5 Retrieved Documents:
1. Title: /teamspace/studios/this_studio/ICD11-Psych/Team_5/code/data/Mood_Anxiety_Stress.pdf
   Page: 79
   Content: 289
Anxiety and fear-related disorders
Separation anxiety disorder
Essential (required) features
• Marked and excessive fear or anxiety about separation from those individuals to whom 
the person is attached (i.e. with whom the individual has a deep emotional bond) is 
required for diagnosis. In chi...

2. Title: /teamspace/studios/this_studio/ICD11-Psych/Team_5/code/data/Mood_Anxiety_Stress.pdf
   Page: 5
   Content: mood disorders and attention deficit hyperactivity disorder can co-occur, and both diagnoses may 
be assigned if the full diagnostic requirements for each are met.
Boundary with prolonged grief disorder
Prolonged grief disorder is a persistent and pervasive grief response following the death of a pa...

3. Title: /teamspace/studios/this_studio/ICD11-Psych/Team_5/code/data/Mood_Anxiety_Stress.pdf
   Page: 102
   Content: circumsta

In [11]:
concatenated_text = concatenate_texts("This is my description: ",query, top_k_texts)
print("\nConcatenated Text (with Query):")
print(concatenated_text)


Concatenated Text (with Query):
This is my description: I am sad, I am unhappy.

Try to do the diagnosis based on the following reference(s):
289
Anxiety and fear-related disorders
Separation anxiety disorder
Essential (required) features
• Marked and excessive fear or anxiety about separation from those individuals to whom 
the person is attached (i.e. with whom the individual has a deep emotional bond) is 
required for diagnosis. In children and adolescents, key attachment figures that are most 
commonly the focus of separation anxiety include parents, caregivers and other family 
members, and the fear or anxiety is beyond what would be considered developmentally 
normative. In adults, separation anxiety most often involves a spouse, romantic partner or 
children. Manifestations of fear or anxiety related to separation depend on the individual’s 
developmental level, but may include:
• persistent thoughts that harm or some other untoward event (e.g. being kidnapped) will 
lead to se

In [25]:
### Get the output from the model
## Load the model
from transformers import T5Tokenizer, T5ForConditionalGeneration
tokenizer = T5Tokenizer.from_pretrained("t5-base")
model = T5ForConditionalGeneration.from_pretrained("t5-base").to(device)

In [26]:
# Tokenize the concatenated text for the model (truncated to 1024 tokens)
inputs = tokenizer(concatenated_text, max_length=512, return_tensors="pt", 
                    truncation=True, padding=True).to(device)

input_ids = inputs["input_ids"]
attention_mask = inputs["attention_mask"]

# Generate the output using the model
with torch.no_grad():
    generated_ids = model.generate(
        input_ids,                         # Pass the input_ids
        attention_mask=attention_mask,     # Pass the attention mask
        max_new_tokens=200,                # Generate 200 new tokens
        num_beams=5,                       # Beam search with 5 beams for better quality
        temperature=0.7,                   # Controls the randomness in the generation
        top_k=50,                          # Top-K sampling to control randomness
        top_p=0.95,                        # Top-p (nucleus) sampling
        early_stopping=True,               # Stops once all beams generate an end-of-sequence token
        no_repeat_ngram_size=3,            # Prevents repetition of trigrams in the generated output
        eos_token_id=tokenizer.eos_token_id # Ensure eos_token_id is set
    )

# Decode the output tokens to get the final generated answer
output = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

# Print the generated answer (diagnosis or response)
print("\nGenerated Answer (Diagnosis):", output)

/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:606: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(



Generated Answer (Diagnosis): separation anxiety disorder is a persistent and pervasive grief response following the death of a partner, parent, child or other person close to the bereaved . it is characterized by longing for the deceased or persistent preoccupation with the deceased accompanied by intense emotional pain (e.g. sadness, guilt, anger, denial, blame, difficulty accepting the death, feeling one has lost a part of one’s self)
